# 🛒 Đồ Án Khai Phá Dữ Liệu - RetailRocket E-Commerce

**Dataset:** RetailRocket E-Commerce Dataset  
**Bài toán:** Dự đoán hành vi mua hàng của người dùng (Purchase Prediction)  
**Phương pháp:** Tiền xử lý dữ liệu + Feature Engineering + Machine Learning

---

## 📋 Mô tả Dataset

| File | Mô tả | Số dòng |
|------|--------|---------|
| `events.csv` | Lịch sử tương tác (view, addtocart, transaction) | ~2.75M |
| `category_tree.csv` | Cây phân cấp danh mục sản phẩm | ~1.67K |
| `item_properties_part1.csv` | Thuộc tính sản phẩm (phần 1) | ~11M |
| `item_properties_part2.csv` | Thuộc tính sản phẩm (phần 2) | ~9.28M |

## 🎯 Mục tiêu
Xây dựng mô hình dự đoán liệu một người dùng có **mua hàng** hay không dựa trên hành vi lướt web của họ.

## 1. Import Thư Viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve,
    accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.pipeline import Pipeline

# Style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
PALETTE = ['#6C63FF', '#FF6584', '#43D8B5', '#FFB347', '#5B9BD5']

print('✅ Thư viện đã được import thành công!')
print(f'📦 Pandas v{pd.__version__} | NumPy v{np.__version__}')

## 2. Tải Dữ Liệu (Data Loading)

In [ ]:
DATA_DIR = './data/'

print('📂 Đang tải dữ liệu...')

events = pd.read_csv(DATA_DIR + 'events.csv')
category_tree = pd.read_csv(DATA_DIR + 'category_tree.csv')

print(f'✅ events.csv       : {events.shape[0]:,} dòng × {events.shape[1]} cột')
print(f'✅ category_tree.csv: {category_tree.shape[0]:,} dòng × {category_tree.shape[1]} cột')
print('\n⚠️  Lưu ý: item_properties (~20M dòng) sẽ chỉ sample một phần để tiết kiệm RAM')

# Sample item_properties để tránh OOM
print('\n📂 Đang tải item_properties (sample 500K dòng mỗi phần)...')
prop1 = pd.read_csv(DATA_DIR + 'item_properties_part1.csv', nrows=500000)
prop2 = pd.read_csv(DATA_DIR + 'item_properties_part2.csv', nrows=500000)
item_props = pd.concat([prop1, prop2], ignore_index=True)
print(f'✅ item_properties   : {item_props.shape[0]:,} dòng × {item_props.shape[1]} cột')

## 3. Khám Phá Dữ Liệu (EDA - Exploratory Data Analysis)

In [ ]:
print('=' * 60)
print('📊 TỔNG QUAN EVENTS.CSV')
print('=' * 60)
print(events.head())
print('\n📐 Thông tin cột:')
events.info()
print('\n📊 Thống kê mô tả:')
display(events.describe())

In [ ]:
# Phân phối loại sự kiện
event_counts = events['event'].value_counts()
print('📊 Phân phối loại sự kiện:')
print(event_counts)
print(f'\nTỉ lệ chuyển đổi (view → addtocart): {event_counts["addtocart"]/event_counts["view"]*100:.2f}%')
print(f'Tỉ lệ chuyển đổi (view → transaction): {event_counts["transaction"]/event_counts["view"]*100:.2f}%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('📊 Phân Tích Sự Kiện Người Dùng', fontsize=15, fontweight='bold', y=1.02)

# 1. Biểu đồ loại sự kiện
ax = axes[0]
bars = ax.bar(event_counts.index, event_counts.values,
               color=PALETTE[:3], edgecolor='white', linewidth=1.5)
ax.set_title('Phân Phối Loại Sự Kiện', fontweight='bold')
ax.set_xlabel('Loại sự kiện')
ax.set_ylabel('Số lượng')
for bar, val in zip(bars, event_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20000,
            f'{val:,}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# 2. Pie chart tỉ lệ
ax = axes[1]
wedges, texts, autotexts = ax.pie(
    event_counts.values, labels=event_counts.index,
    autopct='%1.1f%%', colors=PALETTE[:3],
    startangle=90, pctdistance=0.8
)
for text in autotexts:
    text.set_fontweight('bold')
ax.set_title('Tỉ Lệ Sự Kiện', fontweight='bold')

# 3. Funnel chuyển đổi
ax = axes[2]
stages = ['Lượt xem', 'Thêm giỏ', 'Mua hàng']
values = [event_counts['view'], event_counts['addtocart'], event_counts['transaction']]
colors_funnel = ['#6C63FF', '#FF6584', '#43D8B5']
bars2 = ax.barh(stages[::-1], values[::-1], color=colors_funnel[::-1],
                edgecolor='white', linewidth=1.5, height=0.5)
ax.set_title('Funnel Chuyển Đổi', fontweight='bold')
ax.set_xlabel('Số lượng')
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
for bar, val in zip(bars2, values[::-1]):
    ax.text(bar.get_width() + 10000, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('output_event_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Đã lưu: output_event_distribution.png')

## 4. Tiền Xử Lý Dữ Liệu (Data Preprocessing)

In [ ]:
print('🔧 Bước 1: Xử lý kiểu dữ liệu và timestamp')
print('-' * 50)

# Chuyển timestamp từ milliseconds sang datetime
events['datetime'] = pd.to_datetime(events['timestamp'], unit='ms')
events['date']     = events['datetime'].dt.date
events['hour']     = events['datetime'].dt.hour
events['dayofweek']= events['datetime'].dt.dayofweek  # 0=Mon, 6=Sun
events['month']    = events['datetime'].dt.month

print(f'✅ Thời gian bắt đầu : {events["datetime"].min()}')
print(f'✅ Thời gian kết thúc: {events["datetime"].max()}')
print(f'✅ Khoảng thời gian  : {(events["datetime"].max() - events["datetime"].min()).days} ngày')
print(f'\n✅ Đã thêm cột: datetime, date, hour, dayofweek, month')
events[['timestamp', 'datetime', 'hour', 'dayofweek', 'month']].head(3)

In [ ]:
print('🔧 Bước 2: Kiểm tra và xử lý giá trị thiếu')
print('-' * 50)

missing = events.isnull().sum()
missing_pct = (missing / len(events) * 100).round(2)
missing_df = pd.DataFrame({'Thiếu': missing, 'Tỉ lệ (%)': missing_pct})
print(missing_df)

# transactionid chỉ có khi event='transaction' → bình thường
print('\n📌 Giải thích: transactionid=NaN là bình thường vì')
print('   chỉ sự kiện "transaction" mới có transactionid')

# Tạo cờ has_transaction_id
events['has_transaction'] = events['transactionid'].notna().astype(int)
print('\n✅ Đã tạo cột "has_transaction" (0/1)')

In [ ]:
print('🔧 Bước 3: Kiểm tra trùng lặp')
print('-' * 50)

dup_count = events.duplicated().sum()
print(f'Số dòng trùng lặp hoàn toàn: {dup_count:,}')

# Kiểm tra cùng visitor - item - event trong cùng giây
dup_key = events.duplicated(subset=['visitorid', 'itemid', 'event', 'timestamp']).sum()
print(f'Số dòng trùng (visitor+item+event+timestamp): {dup_key:,}')

if dup_count > 0:
    events = events.drop_duplicates()
    print(f'✅ Đã loại bỏ {dup_count:,} dòng trùng lặp')
else:
    print('✅ Không có dòng trùng lặp hoàn toàn')

print(f'\nSố dòng sau khi làm sạch: {len(events):,}')

In [ ]:
print('🔧 Bước 4: Kiểm tra và xử lý Outliers')
print('-' * 50)

# Số sự kiện mỗi visitor
visitor_event_counts = events.groupby('visitorid').size()
print('Thống kê số sự kiện mỗi visitor:')
print(visitor_event_counts.describe())

# Phát hiện outlier bằng IQR
Q1 = visitor_event_counts.quantile(0.25)
Q3 = visitor_event_counts.quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 3 * IQR

bot_visitors = visitor_event_counts[visitor_event_counts > upper_bound]
print(f'\n⚠️  Bot nghi vấn (>{upper_bound:.0f} sự kiện): {len(bot_visitors):,} visitor')
print(f'   Sự kiện từ bot: {bot_visitors.sum():,} ({bot_visitors.sum()/len(events)*100:.2f}%)')

# Lọc bot
normal_visitors = visitor_event_counts[visitor_event_counts <= upper_bound].index
events_clean = events[events['visitorid'].isin(normal_visitors)].copy()
print(f'\n✅ Dữ liệu sau khi lọc bot: {len(events_clean):,} dòng ({len(events_clean)/len(events)*100:.1f}% giữ lại)')

In [ ]:
# Visualize phân phối sự kiện theo thời gian
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('⏰ Phân Tích Hành Vi Theo Thời Gian', fontsize=15, fontweight='bold')

# 1. Theo giờ
ax = axes[0, 0]
hourly = events_clean.groupby('hour').size()
ax.fill_between(hourly.index, hourly.values, alpha=0.4, color='#6C63FF')
ax.plot(hourly.index, hourly.values, 'o-', color='#6C63FF', linewidth=2)
ax.set_title('Phân Phối Theo Giờ', fontweight='bold')
ax.set_xlabel('Giờ trong ngày')
ax.set_ylabel('Số sự kiện')
ax.set_xticks(range(0, 24, 2))

# 2. Theo ngày trong tuần
ax = axes[0, 1]
days = ['Thứ 2', 'Thứ 3', 'Thứ 4', 'Thứ 5', 'Thứ 6', 'Thứ 7', 'CN']
daily = events_clean.groupby('dayofweek').size()
bars = ax.bar(days, daily.values, color=PALETTE, edgecolor='white', linewidth=1.5)
ax.set_title('Phân Phối Theo Ngày Trong Tuần', fontweight='bold')
ax.set_xlabel('Ngày')
ax.set_ylabel('Số sự kiện')

# 3. Phân phối sự kiện mỗi visitor (sau khi lọc)
ax = axes[1, 0]
visitor_counts_clean = events_clean.groupby('visitorid').size()
ax.hist(visitor_counts_clean, bins=50, color='#43D8B5', edgecolor='white')
ax.set_title('Phân Phối Số Sự Kiện / Visitor', fontweight='bold')
ax.set_xlabel('Số sự kiện')
ax.set_ylabel('Số visitor')
ax.axvline(visitor_counts_clean.median(), color='red', linestyle='--',
           label=f'Trung vị: {visitor_counts_clean.median():.0f}')
ax.legend()

# 4. Heatmap giờ × ngày
ax = axes[1, 1]
heatmap_data = events_clean.groupby(['dayofweek', 'hour']).size().unstack(fill_value=0)
sns.heatmap(heatmap_data, ax=ax, cmap='viridis', fmt='', cbar_kws={'label': 'Số sự kiện'})
ax.set_title('Heatmap Hoạt Động (Ngày × Giờ)', fontweight='bold')
ax.set_xlabel('Giờ')
ax.set_yticklabels(days, rotation=0)

plt.tight_layout()
plt.savefig('output_temporal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Đã lưu: output_temporal_analysis.png')

## 5. Feature Engineering (Xây Dựng Đặc Trưng)

Tạo features dựa trên hành vi của từng **visitor** (người dùng) để dự đoán khả năng mua hàng.

In [ ]:
print('⚙️  Bước 5: Feature Engineering - Xây dựng đặc trưng cho từng visitor')
print('=' * 60)

df = events_clean.copy()

# --- Đặc trưng RFM-style ---
# Recency: khoảng cách lần cuối tương tác
last_event_time = df.groupby('visitorid')['timestamp'].max().rename('last_timestamp')
global_max_ts   = df['timestamp'].max()
recency_ms      = global_max_ts - last_event_time
recency_hours   = (recency_ms / (1000 * 3600)).rename('recency_hours')

# Frequency: tổng số sự kiện
total_events    = df.groupby('visitorid').size().rename('total_events')

# Đặc trưng theo loại sự kiện
event_pivot = df.groupby(['visitorid', 'event']).size().unstack(fill_value=0)
event_pivot.columns = [f'n_{col}' for col in event_pivot.columns]
for col in ['n_view', 'n_addtocart', 'n_transaction']:
    if col not in event_pivot.columns:
        event_pivot[col] = 0

# Số sản phẩm khác nhau đã xem
unique_items    = df.groupby('visitorid')['itemid'].nunique().rename('unique_items_viewed')

# Số ngày hoạt động
active_days     = df.groupby('visitorid')['date'].nunique().rename('active_days')

# Tỉ lệ chuyển đổi
features = pd.concat([total_events, recency_hours, event_pivot, unique_items, active_days], axis=1).fillna(0)
features['atc_rate']   = features['n_addtocart'] / (features['n_view'] + 1)  # add-to-cart rate
features['buy_rate']   = features['n_transaction'] / (features['n_view'] + 1)  # purchase rate
features['item_per_day']= features['unique_items_viewed'] / (features['active_days'] + 1)
features['avg_events_per_day'] = features['total_events'] / (features['active_days'] + 1)

# --- Biến mục tiêu: đã mua hàng hay chưa ---
features['purchased'] = (features['n_transaction'] > 0).astype(int)

print(f'✅ Tổng số visitors: {len(features):,}')
print(f'✅ Số features: {len(features.columns) - 1}')
print(f'\n📊 Phân phối nhãn mục tiêu:')
print(features['purchased'].value_counts())
print(f'Tỉ lệ mua hàng: {features["purchased"].mean()*100:.2f}%')

print('\n📋 Danh sách features:')
feature_cols = [c for c in features.columns if c != 'purchased']
for f in feature_cols:
    print(f'  - {f}')

In [ ]:
# Phân tích features
print('📊 Thống kê mô tả features:')
display(features.describe().T.round(2))

In [ ]:
# So sánh hành vi giữa người mua và không mua
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('🔍 So Sánh Hành Vi: Người Mua vs Không Mua', fontsize=15, fontweight='bold')

plot_features = [
    ('total_events', 'Tổng số sự kiện'),
    ('n_view', 'Số lần xem'),
    ('n_addtocart', 'Số lần thêm giỏ'),
    ('unique_items_viewed', 'Số sản phẩm khác nhau'),
    ('active_days', 'Số ngày hoạt động'),
    ('recency_hours', 'Recency (giờ)'),
]

for (feat, title), ax in zip(plot_features, axes.flatten()):
    data_0 = features[features['purchased'] == 0][feat]
    data_1 = features[features['purchased'] == 1][feat]
    
    # Clip để hiển thị đẹp
    clip_val = data_0.quantile(0.95)
    data_0_c = data_0.clip(upper=clip_val)
    data_1_c = data_1.clip(upper=clip_val)
    
    ax.hist(data_0_c, bins=40, alpha=0.6, color='#6C63FF', label='Không mua', density=True)
    ax.hist(data_1_c, bins=40, alpha=0.6, color='#FF6584', label='Đã mua', density=True)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel('Mật độ')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('output_feature_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Đã lưu: output_feature_comparison.png')

In [ ]:
# Correlation matrix
fig, ax = plt.subplots(figsize=(12, 9))
corr = features[feature_cols + ['purchased']].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, cbar_kws={'shrink': .8})
ax.set_title('🔗 Ma Trận Tương Quan Giữa Các Features', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('output_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Đã lưu: output_correlation_matrix.png')

## 6. Machine Learning - Xây Dựng Mô Hình Dự Đoán

**Bài toán:** Binary Classification — Dự đoán visitor có mua hàng hay không

**Các mô hình so sánh:**
1. Logistic Regression (baseline)
2. Decision Tree
3. Random Forest
4. Gradient Boosting

In [ ]:
print('🤖 Chuẩn bị dữ liệu cho Machine Learning')
print('=' * 60)

X = features[feature_cols].copy()
y = features['purchased'].copy()

# Train/Test split (stratified để giữ tỉ lệ nhãn)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'✅ Tập train: {X_train.shape[0]:,} mẫu')
print(f'✅ Tập test : {X_test.shape[0]:,} mẫu')
print(f'\n📊 Phân phối nhãn trong train:')
print(y_train.value_counts(normalize=True).round(4))
print(f'\n📊 Phân phối nhãn trong test:')
print(y_test.value_counts(normalize=True).round(4))

# Scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print('\n✅ Đã chuẩn hóa features với StandardScaler')

In [ ]:
print('🏋️  Huấn luyện các mô hình...')
print('=' * 60)

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced', random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=8, min_samples_leaf=50, class_weight='balanced', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=30,
        class_weight='balanced', random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=150, max_depth=5, learning_rate=0.1, random_state=42
    ),
}

results = {}

for name, model in models.items():
    print(f'\n⏳ Đang huấn luyện: {name}...')
    
    # Dùng scaled data cho Logistic Regression, raw data cho tree-based
    if 'Logistic' in name:
        X_tr, X_te = X_train_scaled, X_test_scaled
    else:
        X_tr, X_te = X_train.values, X_test.values
    
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'accuracy' : accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall'   : recall_score(y_test, y_pred, zero_division=0),
        'f1'       : f1_score(y_test, y_pred, zero_division=0),
        'roc_auc'  : roc_auc_score(y_test, y_prob),
    }
    print(f'  ✅ Accuracy: {results[name]["accuracy"]:.4f} | F1: {results[name]["f1"]:.4f} | AUC: {results[name]["roc_auc"]:.4f}')

print('\n🎉 Hoàn thành huấn luyện tất cả mô hình!')

In [ ]:
# Bảng so sánh kết quả
metrics_df = pd.DataFrame({
    name: {
        'Accuracy' : res['accuracy'],
        'Precision': res['precision'],
        'Recall'   : res['recall'],
        'F1-Score' : res['f1'],
        'ROC-AUC'  : res['roc_auc'],
    }
    for name, res in results.items()
}).T.round(4)

print('📊 BẢNG SO SÁNH HIỆU NĂNG CÁC MÔ HÌNH:')
print('=' * 70)
display(metrics_df.style.highlight_max(color='#90EE90').format('{:.4f}'))

best_model_name = metrics_df['ROC-AUC'].idxmax()
print(f'\n🏆 Mô hình tốt nhất (theo ROC-AUC): {best_model_name}')

## 7. Đánh Giá Mô Hình (Model Evaluation)

In [ ]:
# So sánh metrics bằng biểu đồ
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('📊 So Sánh Hiệu Năng Mô Hình', fontsize=15, fontweight='bold')

# 1. Biểu đồ grouped bar
ax = axes[0]
metrics_plot = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x  = np.arange(len(metrics_plot))
w  = 0.2
offsets = np.linspace(-0.3, 0.3, 4)

for i, (name, color) in enumerate(zip(results.keys(), PALETTE)):
    vals = [metrics_df.loc[name, m] for m in metrics_plot]
    bars = ax.bar(x + offsets[i], vals, w, label=name, color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(metrics_plot)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Tất Cả Metrics', fontweight='bold')
ax.legend(loc='upper right', fontsize=8)
ax.axhline(0.5, linestyle='--', color='gray', alpha=0.5, label='Baseline 0.5')

# 2. ROC Curves
ax = axes[1]
if 'Logistic' in results:
    X_te_lg = X_test_scaled
else:
    X_te_lg = X_test.values

for (name, res), color in zip(results.items(), PALETTE):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    auc = res['roc_auc']
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Baseline')
ax.fill_between([0, 1], [0, 1], alpha=0.1, color='gray')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])

plt.tight_layout()
plt.savefig('output_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Đã lưu: output_model_comparison.png')

In [ ]:
# Confusion matrices cho tất cả mô hình
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
fig.suptitle('🔲 Confusion Matrix - Tất Cả Mô Hình', fontsize=14, fontweight='bold')

labels = ['Không mua', 'Đã mua']

for ax, (name, res), color in zip(axes, results.items(), PALETTE):
    cm = confusion_matrix(y_test, res['y_pred'])
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    sns.heatmap(
        cm, annot=True, fmt='d', ax=ax,
        cmap=sns.light_palette(color, as_cmap=True),
        xticklabels=labels, yticklabels=labels,
        cbar=False
    )
    # Thêm tỉ lệ %
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.7, f'({cm_norm[i,j]*100:.1f}%)',
                    ha='center', va='center', fontsize=8, color='gray')
    ax.set_title(f'{name}\nF1={res["f1"]:.3f}', fontweight='bold', fontsize=9)
    ax.set_xlabel('Dự đoán')
    ax.set_ylabel('Thực tế')

plt.tight_layout()
plt.savefig('output_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Đã lưu: output_confusion_matrices.png')

In [ ]:
# Feature Importance từ mô hình tốt nhất
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🔍 Feature Importance Analysis', fontsize=14, fontweight='bold')

# Random Forest feature importance
rf_model = results['Random Forest']['model']
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

ax = axes[0]
colors_imp = plt.cm.viridis(np.linspace(0.2, 0.9, len(importances)))
bars = ax.barh(importances.index, importances.values, color=colors_imp, edgecolor='white')
ax.set_title('Random Forest Feature Importance', fontweight='bold')
ax.set_xlabel('Importance Score')
for bar, val in zip(bars, importances.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)

# Gradient Boosting feature importance
gb_model = results['Gradient Boosting']['model']
importances_gb = pd.Series(gb_model.feature_importances_, index=feature_cols).sort_values(ascending=True)

ax = axes[1]
colors_imp2 = plt.cm.plasma(np.linspace(0.2, 0.9, len(importances_gb)))
bars2 = ax.barh(importances_gb.index, importances_gb.values, color=colors_imp2, edgecolor='white')
ax.set_title('Gradient Boosting Feature Importance', fontweight='bold')
ax.set_xlabel('Importance Score')
for bar, val in zip(bars2, importances_gb.values):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('output_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 Đã lưu: output_feature_importance.png')

In [ ]:
# Phân tích chi tiết mô hình tốt nhất
print(f'\n📑 CLASSIFICATION REPORT - {best_model_name}')
print('=' * 60)
best_res = results[best_model_name]
print(classification_report(y_test, best_res['y_pred'],
                            target_names=['Không mua', 'Đã mua'],
                            digits=4))

In [ ]:
# Cross-validation cho mô hình tốt nhất
print(f'🔄 Cross-Validation (5-fold) - {best_model_name}')
print('=' * 60)

best_model = results[best_model_name]['model']
X_cv = X_train_scaled if 'Logistic' in best_model_name else X_train.values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X_cv, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f'AUC per fold: {[f"{s:.4f}" for s in cv_scores]}')
print(f'\nMean AUC : {cv_scores.mean():.4f}')
print(f'Std  AUC : {cv_scores.std():.4f}')
print(f'Min  AUC : {cv_scores.min():.4f}')
print(f'Max  AUC : {cv_scores.max():.4f}')

## 8. Tổng Kết & Kết Luận

In [ ]:
print('📋 TỔNG KẾT ĐỒ ÁN KHAI PHÁ DỮ LIỆU')
print('=' * 70)
print()
print('📂 DỮ LIỆU:')
print(f'   • Tổng sự kiện     : {len(events):,}')
print(f'   • Tổng visitors    : {events["visitorid"].nunique():,}')
print(f'   • Tổng sản phẩm    : {events["itemid"].nunique():,}')
print(f'   • Khoảng thời gian : {(events["datetime"].max() - events["datetime"].min()).days} ngày')
print()
print('🧹 TIỀN XỬ LÝ:')
print(f'   • Xử lý timestamp → datetime features')
print(f'   • Phát hiện & loại bỏ bot ({events_clean.shape[0]:,} dòng sạch)')
print(f'   • Không có missing values cần xử lý (trừ transactionid là bình thường)')
print()
print('⚙️  FEATURE ENGINEERING:')
print(f'   • {len(feature_cols)} features RFM-style được tạo ra')
print(f'   • n_view, n_addtocart, n_transaction, unique_items, active_days')
print(f'   • recency_hours, atc_rate, buy_rate, item_per_day, avg_events_per_day')
print()
print('🤖 KẾT QUẢ MACHINE LEARNING:')
print()
display(metrics_df.style.highlight_max(color='#90EE90').format('{:.4f}'))
print()
print(f'🏆 Mô hình tốt nhất: {best_model_name}')
print(f'   ROC-AUC = {results[best_model_name]["roc_auc"]:.4f}')
print(f'   F1-Score = {results[best_model_name]["f1"]:.4f}')
print()
print('💡 NHẬN XÉT:')
print('   1. Dataset có class imbalance cao (~98% không mua)')
print('   2. Feature n_transaction và n_addtocart là quan trọng nhất')
print('   3. Gradient Boosting / Random Forest cho kết quả tốt nhất')
print('   4. Hành vi add-to-cart là dấu hiệu mạnh nhất dự đoán mua hàng')
print()
print('📁 CÁC FILE ĐẦU RA:')
outputs = [
    'output_event_distribution.png',
    'output_temporal_analysis.png',
    'output_feature_comparison.png',
    'output_correlation_matrix.png',
    'output_model_comparison.png',
    'output_confusion_matrices.png',
    'output_feature_importance.png',
]
for f in outputs:
    print(f'   ✅ {f}')